#### [fish.csv SVC 분류] <hr>

- 데이터 : fish.csv (Bream vs Smelt)
- 목 적 : Bream(도미) / Smelt(빙어) 이진 분류
- 학습방법 : SVC (Support Vector Classifier)
- 전처리 : RobustScaler + Pipeline

In [6]:
## 데이터 관련
import pandas as pd
import numpy as np

## 시각화 관련
import matplotlib.pyplot as plt, koreanize_matplotlib
import seaborn as sns

## ML 교차검증 관련
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.model_selection import StratifiedKFold

## ML 데이터 전처리 관련
from sklearn.preprocessing import MinMaxScaler, RobustScaler

## ML 모델 관련
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.pipeline import Pipeline

import joblib, os

In [23]:
df = pd.read_csv('../Data/Numbers/fish.csv')
df = df[df['Species'].isin(['Bream', 'Smelt'])].reset_index(drop=True)
print(df['Species'].value_counts())

df['Species'] = df['Species'].astype('category')
df.head(3)

Species
Bream    35
Smelt    14
Name: count, dtype: int64


,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961


In [25]:
feature = df[['Weight',"Height","Width"]]
target = df[df.columns[0]]

In [12]:
x_train, x_test, y_train, y_test = train_test_split(feature, 
                                                    target,
                                                    test_size=0.2,
                                                    random_state=10,
                                                    stratify=target)

**[3] 파이프라인 + 교차 검증 + 튜닝 준비**

In [ ]:
# 파이프라인 인스턴스 생성 : 전처리기(스케일러) + 모델
pipe = Pipeline(steps=[
    ('rbScaler', RobustScaler()),
    ('svcMdel',  SVC(class_weight='balanced'))
])

# 분류용 교차검증 인스턴스 생성
skFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)

# 모델 하이퍼파라미터 설정 : 파이프라인 단계이름__파라미터명 (__ 더블언더스코어)
params = {
    'svcMdel__C':      [0.1, 0.5, 1.0, 5, 10],
    'svcMdel__kernel': ['rbf', 'linear', 'poly']
}

# 교차검증 + 튜닝 인스턴스 생성 (다중 scoring, refit 기준 지표 지정)
tunning = GridSearchCV(pipe, cv=skFold, param_grid=params,
                       scoring=['accuracy', 'f1_macro', 'f1_weighted'],
                       refit='f1_macro',
                       return_train_score=True,
                       n_jobs=-1)

[4] 교차검증 + 튜닝 진행

In [ ]:
tunning.fit(x_train, y_train)

**[5] 교차검증+튜닝결과**

In [ ]:
print(f'best_params_ : { tunning.best_params_}')
print(f'best_score_ : { tunning.best_score_}')
best_model = tunning.best_estimator_

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 테스트 데이터 예측
y_pred = best_model.predict(x_test)

# Train / Test 점수
print(f'Train Score : {best_model.score(x_train, y_train):.4f}')
print(f'Test  Score : {best_model.score(x_test,  y_test):.4f}')
print()

# 클래스별 분류 리포트
print(classification_report(y_test, y_pred))

# Confusion Matrix 시각화
cm   = confusion_matrix(y_test, y_pred)
cmDF = pd.DataFrame(cm, index=['Bream','Smelt'], columns=['Bream','Smelt'])

plt.figure(figsize=(5, 4))
sns.heatmap(cmDF, annot=True, fmt='d', cmap='Blues')
plt.xlabel('예측')
plt.ylabel('실제')
plt.title('SVC Confusion Matrix')
plt.show()